# AI Workforce Intelligence, Phase 5: the dashboard (CS3216 Time Series Analysis and Forecasting)

Phase 5 is a plain-language dashboard that shows the project's real results and lets anyone try their own numbers through the Phase 4 engine. Streamlit does not run inside Colab, so this notebook shows the pages as screenshots and the numbers behind them.

> **Where this fits.** This is **Phase 5** of 6.
> **Needs:** the saved Phase 4 results and the engine (inside the zip).
> **Produces:** `src/dashboard/` (run with `streamlit run src/dashboard/app.py`) and `outputs/phase5/` (screenshots, notes).

> **How to use this notebook**
> 1. In Colab: *Runtime > Run all*. When asked, upload `ai-workforce-intel_colab.zip` (the same zip for all three notebooks).
> 2. `MODE = "show"` (default) only displays the finished results inside the zip: under 1 minute.
> 3. `MODE = "run"` recomputes **this phase only** (nothing to recompute (the dashboard runs locally with Streamlit) on Colab), using the earlier phases' results in the zip.
> 4. Outside Colab, put the unzipped `ai-workforce-intel` folder (or the zip) next to this notebook.

In [ ]:
# ---- Settings: change these, then Runtime > Run all ----
MODE = "show"       # "show": display the finished results in the bundle (under 1 minute)
                    # "run":  recompute Phase 5 first, then display the new results
RUN_FAST = True     # only for MODE = "run": True = --fast, False = full run
DRIVERS = None      # None = all drivers, or a list such as ["burst_tokens_total_daily"]
PHASE = 5

import os
MODE = os.environ.get("TSAF_MODE", MODE)  # used only by the automated test; ignore in Colab
print("Phase", PHASE, "| MODE =", MODE, "| RUN_FAST =", RUN_FAST, "| DRIVERS =", DRIVERS or "all")

### Helper functions
Small helpers used by every step: `step` turns any problem into a one-line message (so *Run all*
never stops), `show_img` and `show_table` display figures and tables at a readable size, and
`show_source` prints a real function from the project.

In [ ]:
import contextlib, importlib, inspect, json, re, subprocess, sys, time, warnings
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_colwidth", 90)

@contextlib.contextmanager
def step(name):
    """Run one notebook step; a problem prints a short message instead of a traceback."""
    try:
        yield
    except FileNotFoundError as exc:
        print(f"[{name}] skipped: a result file is missing ({Path(str(exc.filename or exc)).name}).")
    except Exception as exc:
        print(f"[{name}] skipped: {type(exc).__name__}: {str(exc)[:200]}")

def show_img(path, width=900):
    """Display a saved PNG figure, or a short message if it does not exist."""
    path = Path(path)
    if not path.exists():
        print(f"No figure {path.name}; skipped.")
        return
    display(Image(filename=str(path), width=width))

def show_table(df, n=10, cols=None, digits=3):
    """Display the first n rows of a table, selected columns, rounded numbers."""
    if df is None or len(df) == 0:
        print("No rows to show.")
        return
    df = df[[c for c in cols if c in df.columns]] if cols else df
    display(df.head(n).round(digits).reset_index(drop=True))

def read_table(phase, name):
    """Read outputs/<phase>/tables/<name>.csv."""
    return pd.read_csv(Path("outputs") / phase / "tables" / f"{name}.csv")

def md_section(path, title):
    """Text of the '## ... title ...' section of a generated summary file (None if absent)."""
    text = Path(path).read_text(encoding="utf-8")
    pat = r"^## [^\n]*" + re.escape(title) + r"[^\n]*\n(.*?)(?=^## |\Z)"
    m = re.search(pat, text, flags=re.M | re.S)
    return m.group(1).strip() if m else None

def show_source(module, func, max_lines=25):
    """Print the source of module.func (first max_lines lines), or a short message."""
    try:
        obj = getattr(importlib.import_module(module), func)
        lines = inspect.getsource(obj).splitlines()
    except Exception as exc:
        print(f"Could not find {module}.{func}: {type(exc).__name__}; skipped.")
        return
    more = f"\n# ... {len(lines) - max_lines} more lines" if len(lines) > max_lines else ""
    display(Markdown(f"`{module}.{func}`\n```python\n" + "\n".join(lines[:max_lines]) + more + "\n```"))

## Setup 1: get the project
If the project folder is not here yet, the cell asks you to upload `ai-workforce-intel_colab.zip`
(about 10 MB), unzips it and moves into the folder. Outside Colab it looks for the folder (or the zip)
next to the notebook. The commented lines show how to unzip it from Google Drive instead.

In [ ]:
def _is_project(p):
    return (p / "src" / "run_phase2.py").exists()

def _find_project():
    here = Path.cwd()
    for c in (here, here / "ai-workforce-intel", Path("/content/ai-workforce-intel")):
        if _is_project(c):
            return c
    return None

def _get_zip():
    if os.environ.get("TSAF_BUNDLE_PATH"):          # automated test: no upload dialog
        return Path(os.environ["TSAF_BUNDLE_PATH"])
    if Path("ai-workforce-intel_colab.zip").exists():
        return Path("ai-workforce-intel_colab.zip")
    try:
        from google.colab import files               # only exists inside Colab
    except ImportError:
        print("Not in Colab: place ai-workforce-intel_colab.zip or the unzipped folder next to this notebook.")
        return None
    print("Please upload ai-workforce-intel_colab.zip")
    uploaded = files.upload()
    return Path(next(iter(uploaded))) if uploaded else None

# Alternative: unzip from Google Drive instead of uploading (uncomment the two lines):
# from google.colab import drive; drive.mount("/content/drive")
# import zipfile; zipfile.ZipFile("/content/drive/MyDrive/ai-workforce-intel_colab.zip").extractall("/content")

PROJECT = None
with step("get the project"):
    PROJECT = _find_project()
    if PROJECT is None:
        zpath = _get_zip()
        if zpath is not None and zpath.exists():
            import zipfile
            base = Path("/content") if Path("/content").is_dir() else Path.cwd()
            zipfile.ZipFile(zpath).extractall(base)
            PROJECT = base / "ai-workforce-intel"
    if PROJECT is not None and _is_project(PROJECT):
        os.chdir(PROJECT)
        if str(PROJECT / "src") not in sys.path:
            sys.path.insert(0, str(PROJECT / "src"))
        print("Project ready: working folder is", Path.cwd().name)
    else:
        print("Project not found. Upload the zip and run this cell again.")

## Setup 2: packages
Colab already has pandas, numpy, statsmodels, scikit-learn, matplotlib and pyarrow. This cell installs
only what is missing from `requirements.txt`, plus `xgboost` if it is missing. Prophet and torch are
optional and are never installed automatically.

In [ ]:
IMPORT_NAME = {"scikit-learn": "sklearn"}
with step("packages"):
    wanted = []
    for line in Path("requirements.txt").read_text().splitlines():
        line = line.split("#")[0].strip()
        if line:
            wanted.append(re.split(r"[<>=!~ ]", line)[0])
    missing = []
    for pkg in wanted + ["xgboost"]:
        try:
            importlib.import_module(IMPORT_NAME.get(pkg, pkg))
        except ImportError:
            missing.append(pkg)
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *missing], check=False)
    else:
        print("All required packages are already installed.")
    import numpy, statsmodels, sklearn, matplotlib
    print(f"Python {sys.version.split()[0]} | pandas {pd.__version__} | numpy {numpy.__version__} | "
          f"statsmodels {statsmodels.__version__} | scikit-learn {sklearn.__version__} | matplotlib {matplotlib.__version__}")
    import optional
    for name, ok in [("xgboost", optional.has_xgboost()), ("prophet", optional.has_prophet()), ("torch (LSTM)", optional.has_torch())]:
        print(f"  optional {name:13s}: {'available' if ok else 'not installed (model skipped)'}")

## Setup 3: seeds and project layout
All randomness uses seed 42, so a rerun gives the same numbers.

In [ ]:
with step("seeds and tree"):
    import random, numpy as np
    random.seed(42)
    np.random.seed(42)
    for top in sorted(p for p in Path(".").iterdir() if p.is_dir() and not p.name.startswith(".")):
        subs = sorted(q.name for q in top.iterdir() if q.is_dir() and q.name != "__pycache__")
        print(f"{top.name}/" + (f"  ->  {', '.join(subs)}" if subs else ""))

## Run
This phase has nothing to recompute in a notebook: the dashboard is a Streamlit app that runs on your own
computer. `MODE = "run"` behaves like `"show"` here.

In [ ]:
print("MODE =", MODE, "- nothing to recompute in this notebook; showing the saved screenshots and results.")

## 1. The dashboard and how to open it
The dashboard is a small website that runs on your own computer. From the project folder:

```bash
pip install -r requirements.txt -r requirements-dashboard.txt
streamlit run src/dashboard/app.py
```

It opens at `http://localhost:8501`. It is written for readers with no technical background: plain words, the
example company is labelled on every page, and the benefit is always called an estimate.

## 2. A tour of the five pages
Screenshots taken from the running dashboard at desktop width (UI v3: pages listed in a side panel on the left, amounts in Indian rupees).

### The big picture
The headline: what the programme costs over a year, the estimated benefit, the running balance and when it pays for itself, for the expected outlook, plus the running balance of all three outlooks and three findings.

In [ ]:
with step("overview"):
    show_img("outputs/phase5/ui_v3/after/overview_1440_light.png", width=900)

### Where the money goes
Monthly cost by type, each tagged Based on trends or Our estimate, the share of each type and the estimated benefit against the cost, with selectors for the outlook and how cautious to be about the future.

In [ ]:
with step("costs"):
    show_img("outputs/phase5/ui_v3/after/costs_1440_light.png", width=900)

### When it pays for itself
A one-sentence verdict, the month the programme pays for itself across many possible futures (with a separate bar for not within 12 months) and the three outlooks side by side.

In [ ]:
with step("timing"):
    show_img("outputs/phase5/ui_v3/after/timing_1440_light.png", width=900)

### What matters most
Which choices change the year-end balance most when they are 20% lower or higher, and what could make the expected outlook pay for itself.

In [ ]:
with step("levers"):
    show_img("outputs/phase5/ui_v3/after/levers_1440_light.png", width=900)

### Try your own numbers
A control panel: change the figures, press Update results, and the real engine reruns on them in a couple of seconds; the results can be downloaded.

In [ ]:
with step("your_company"):
    show_img("outputs/phase5/ui_v3/after/your_company_1440_light.png", width=900)

### On a phone
The same first page at phone width: cards stack and the charts resize.

In [ ]:
with step("phone"):
    show_img("outputs/phase5/ui_v3/after/overview_390_light.png", width=320)

## 2b. The look: light and dark, depth and motion
The pages are listed in a side panel on the left; at its foot are the **Word guide**, a **Dark mode** switch and
**Reduce motion** (or the computer's own reduced-motion setting), which swaps every animation and 3D piece for a
still version. Three 3D pieces remain: a slowly turning shape in the Welcome banner, 3D columns of the year-end
running balance (the same figures are plain bars beside them) and a tilted cost donut (a flat view is one click
away). They show the same saved numbers as the rest of the page. Before and after, and the dark version:

In [ ]:
with step("before and after"):
    show_img("outputs/phase5/ui_v3/before/overview_1440_light.png", width=440)
    show_img("outputs/phase5/ui_v3/after/overview_1440_light.png", width=440)
    show_img("outputs/phase5/ui_v3/after/overview_1440_dark.png", width=900)

## 3. The numbers behind the main cards
The four cards on the first page are read straight from the saved Phase 4 tables and formatted the same way as
in the dashboard: converted to Indian rupees at the one rate in `src/dashboard/currency.py`. Nothing is typed in by hand.

In [ ]:
with step("card numbers"):
    from dashboard import theme as T
    tot = read_table("phase4", "totals_summary")
    be = read_table("phase4", "break_even_summary")
    t = tot[(tot["scenario"] == "expected") & (tot["multiplier"] == 1.0)].iloc[0]
    b = be[(be["scenario"] == "expected") & (be["multiplier"] == 1.0)].iloc[0]
    pays = b["p50_month"] if b["prob_break_even_within_T"] >= 0.5 else float("nan")
    rows = [("Total cost over the year", T.money(t["total_cost_p50"]), "totals_summary.csv", "total_cost_p50"),
            ("Estimated benefit over the year", T.money(t["total_value_p50"]), "totals_summary.csv", "total_value_p50"),
            ("Running balance at year end", T.money(t["cum_net_T_p50"]), "totals_summary.csv", "cum_net_T_p50"),
            ("Pays for itself", T.month(pays, int(b["T"])), "break_even_summary.csv", "p50_month, prob_break_even_within_T")]
    show_table(pd.DataFrame(rows, columns=["card", "shown as", "file", "column"]))

## 4. The "Try your own numbers" idea
The last page sends the viewer's figures to the same engine as Phase 4. Here it runs once with the example
company and once with twice as many employees (2,000 possible futures each).

In [ ]:
with step("your company"):
    import logging
    import numpy as np
    logging.getLogger("engine").setLevel(logging.ERROR)
    import config, config_phase4 as P4
    from dashboard import theme as T
    from engine.inputs import load_inputs
    from engine.drivers import load_drivers
    from engine.simulate import run_engine
    base, _ = load_inputs(P4.EXAMPLE_INPUTS)
    drv = load_drivers(int(base["horizon_months"]), P4.PHASE3_DIR, config.DATA_DIR)
    sc = pd.read_csv(P4.PHASE3_DIR / "forecasts" / "scenario_tables.csv")
    rows = []
    for label, inp in (("Example company", base), ("Twice as many employees", dict(base, employees=2 * base["employees"]))):
        r = run_engine(inp, "expected", 1.0, 2000, 42, drv, sc)
        rows.append({"case": label, "total cost": T.money(np.median(r.total_cost.sum(axis=1))),
                     "estimated benefit": T.money(np.median(r.value.sum(axis=1))),
                     "running balance at the end": T.money(np.median(r.cum_net_T))})
    show_table(pd.DataFrame(rows))

# Summary

The talking points and limitations below are taken directly from `PHASE5_SUMMARY.md`,
so every number comes from the run.

In [ ]:
with step("summary"):
    for title in ("What to say in the review", "Limitations"):
        text = md_section("outputs/phase5/PHASE5_SUMMARY.md", title)
        display(Markdown(f"### Phase 5: {title}\n\n" + (text or "_Section not found._")))

In [ ]:
with step("what we have"):
    display(Markdown("### What we have so far"))
    display(pd.DataFrame([{"phase": "1. exploration and baselines", "built": "loaders, decomposition, trend methods, ADF/KPSS, ACF/PACF, anomaly screen, 5 baselines, backtest", "outputs": "outputs/phase1: tables, figures, handoff.json, PHASE1_SUMMARY.md", "notebook": "TSAF_Phase1.ipynb"}, {"phase": "2. forecasting models and rolling backtests", "built": "smoothing, ARIMA/SARIMA/SARIMAX, Ridge/RF/XGBoost, ensemble, intervals, diagnostics, model choice", "outputs": "outputs/phase2: tables, figures, chosen_models.json, cv_residuals.parquet, PHASE2_SUMMARY.md", "notebook": "TSAF_Phase2.ipynb"}, {"phase": "3. simulated paths and the standard forecast table", "built": "refit of the chosen models, 200 bootstrap paths per driver, holdout interval check, standard table, scenarios", "outputs": "outputs/phase3: forecasts/*.parquet and .csv, manifest.json, inputs_for_phase4.json, PHASE3_SUMMARY.md", "notebook": "TSAF_Phase3.ipynb"}, {"phase": "4. Monte Carlo cost and value engine", "built": "cost lines (tokens, hosting, enablement, integration), value projection, break-even, multiplier, tornado", "outputs": "outputs/phase4: tables (break-even, totals, monthly), figures, assumptions.json, PHASE4_SUMMARY.md", "notebook": "TSAF_Phase4.ipynb"}, {"phase": "5. the dashboard", "built": "five-page plain-language dashboard on the saved results, with a live try-your-own-numbers page", "outputs": "src/dashboard/, outputs/phase5: screenshots, DESIGN_NOTES.md, PHASE5_SUMMARY.md", "notebook": "TSAF_Phase5.ipynb"}]))

**Next: Phase 6, the report**, which writes up the method, the results and their limits.